In [1]:
from openai import OpenAI
import os
from tinydb import TinyDB

In [2]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
) 
model = "openai/gpt-oss-120b"

In [3]:
from hypotheses.HypothesesRefiner import HypothesesRefiner

hypothesesDb = TinyDB("../store/frozenlake/hypotheses.json")
 
envCurator = HypothesesRefiner(client=client, model=model, hypothesesDb=hypothesesDb)

debugTrajectory = '''
== Step 1 ==
== Response: 
== Num_Tools: 1
== Tool: move_down
== Tool Parameters: {}
== New State:
  S  F  F  F 
[F] F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 
======
== Step 2 ==
== Response: 
== Step 3 ==
== Response: 
== Num_Tools: 1
== Tool: move_right
== Tool Parameters: {}
== New State:
 [S] F  F  F 
 F  F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 
======
== Step 4 ==
== Response: Proceeding towards the nearest H by moving down.
== Num_Tools: 1
== Tool: move_down
== Tool Parameters: {}
== New State:
  S  F  F  F 
[F] F  F  H 
 H  F  F  H 
 F  H  F  F 
 F  F  F  G 
======
== Step 5 ==
== Response: Moving right towards the goal while avoiding H.
== Num_Tools: 1
== Tool: move_right
== Tool Parameters: {}
== New State:
  S  F  F  F 
 F  F  F  H 
[H] F  F  H 
 F  H  F  F 
 F  F  F  G 
======
'''

envCurator.run(debugTrajectory, debug=True)

== Response: 
== Num_Tools: 1
== Tool: MODIFY
== Tool Parameters: {
  "bullet_id": "78708830-64de-4f4b-abe3-d6b12afb6e91",
  "content": "The agent moves one cell per action orthogonally (up, down, left, right) and cannot move off the grid. It may move onto any traversable cell type, including free (F), start (S), hazard (H), or goal (G) cells."
}


''

In [4]:
from environments.FrozenLakeEnvironment import FrozenLakeEnvironment
from emulator.FrozenLakeEnvEmulator import FrozenLakeEnvEmulator

hypothesesDb = TinyDB("../store/frozenlake/hypotheses.json")
policyDb = TinyDB("../store/frozenlake/policies.json")

env = FrozenLakeEnvironment(policyDb=policyDb, hypothesesDb=hypothesesDb)
env.reset()
print(env.getGeneratorPrompt())
print(env.getGeneratorTools())
env.moveLeft()
print(env.getState())
print(env.moveDown())
print(env.getGeneratorPrompt())
print(env.getReflectorPrompt("DEBUG"))
print(env.getCuratorPrompt("DEBUG_TRAJECTORY", "DEBUG_REFLECTION"))
print(env.getCuratorTools())


emulatorNoLlm = FrozenLakeEnvEmulator(env) 
emulatorNoLlm.moveDown()
print("EMULATOR NO LLM STATE")
print(emulatorNoLlm.getState())
print("ACTUAL ENV STATE")
print(env.getState())
print("####")

emulatorLlm = FrozenLakeEnvEmulator(env, hypothesesDb=hypothesesDb, client=client, model=model, useLlm=True)
print(emulatorLlm.moveDown())
print("EMULATOR LLM STATE")
print(emulatorLlm.getState())
print("ACTUAL ENV STATE")
print(env.getState())
print("####")

[{'role': 'system', 'content': "\n                     You are a multi-turn LLM Agent Navigator in a dynamic 2D environment. Your goal: reach position G from current position [] using the provided tools for movement.\n\n                     ## Multi-Turn Operation\n                     You receive a NAVIGATION TRAJECTORY containing:\n                     - All your previous decisions and reasoning\n                     - Environment responses after each action\n                     - Current state resulting from your last move\n\n                     Your output becomes input for your next iteration. Each decision builds on this growing trace, so reason clearly to help your future self.\n\n                     ## Core Task\n                     Read the playbook and the reflection -> Apply rules, knowledge and strategies retrieved from those documents -> Decide the next one move from the given context\n\n                     ## Decision Process\n                     1. **Apply Learning

In [5]:
from environments.FrozenLakeEnvironment import FrozenLakeEnvironment
from emulator.FrozenLakeEnvEmulator import FrozenLakeEnvEmulator
import gymnasium as gym

hypothesesDb = TinyDB("../store/frozenlake/hypotheses.json")
policyDb = TinyDB("../store/frozenlake/policies.json")


big_map = [
    "SFFFFFHFFFF",
    "FFFFFHFFFFF",
    "FFFFFFHFHFF",
    "FFFFFFFFFFF",
    "FFFFFFFFFFF",
    "FFFFFFFFFFF",
    "FFHFFFFFFFF",
    "FFFFFFFFFHF",
    "FFFFFFFFFFF",
    "FFFFFFFHFFF",
    "FFFFFFFFFFG"
]
medium_map = [
    "SFFFFHF",
    "FFFFFHF",
    "FHFFFFH",
    "FFFFFFF",
    "FFFHFFF",
    "FFHFFFG"
    ]
small_map = [
    "SFFF",
    "FFFH",
    "HFFH",
    "FHFF",
    "FFFG"
]   
env = FrozenLakeEnvironment(env = gym.make("FrozenLake-v1",
                      render_mode="ansi", 
                      desc=big_map,
                      map_name=None,
                      is_slippery=False,
                      success_rate=0.7,
                      reward_schedule=(1, 0, 0)), policyDb=policyDb, hypothesesDb=hypothesesDb)
env.reset()


emulatorNoLlm = FrozenLakeEnvEmulator(env) 
emulatorNoLlm.moveDown()
print("EMULATOR NO LLM STATE")
print(emulatorNoLlm.getState())
print("ACTUAL ENV STATE")
print(env.getState())
print("####")

emulatorLlm = FrozenLakeEnvEmulator(env, hypothesesDb=hypothesesDb, client=client, model=model, useLlm=True)
emulatorLlm.moveDown()
print(emulatorLlm.getState())
emulatorLlm.moveLeft()
print(emulatorLlm.getState())
emulatorLlm.moveRight()
print(emulatorLlm.getState())
print("EMULATOR LLM STATE")
print(emulatorLlm.getState())
print("ACTUAL ENV STATE")
print(env.getState())
print("####")

EMULATOR NO LLM STATE
 S  F  F  F  F  F  H  F  F  F  F 
[F] F  F  F  F  H  F  F  F  F  F 
 F  F  F  F  F  F  H  F  H  F  F 
 F  F  F  F  F  F  F  F  F  F  F 
 F  F  F  F  F  F  F  F  F  F  F 
 F  F  F  F  F  F  F  F  F  F  F 
 F  F  H  F  F  F  F  F  F  F  F 
 F  F  F  F  F  F  F  F  F  H  F 
 F  F  F  F  F  F  F  F  F  F  F 
 F  F  F  F  F  F  F  H  F  F  F 
 F  F  F  F  F  F  F  F  F  F  G 
ACTUAL ENV STATE
[S] F  F  F  F  F  H  F  F  F  F 
 F  F  F  F  F  H  F  F  F  F  F 
 F  F  F  F  F  F  H  F  H  F  F 
 F  F  F  F  F  F  F  F  F  F  F 
 F  F  F  F  F  F  F  F  F  F  F 
 F  F  F  F  F  F  F  F  F  F  F 
 F  F  H  F  F  F  F  F  F  F  F 
 F  F  F  F  F  F  F  F  F  H  F 
 F  F  F  F  F  F  F  F  F  F  F 
 F  F  F  F  F  F  F  H  F  F  F 
 F  F  F  F  F  F  F  F  F  F  G 
####
S  F  F  F  F  F  H  F  F  F  F
[F]  F  F  F  F  H  F  F  F  F  F
F  F  F  F  F  F  H  F  H  F  F
F  F  F  F  F  F  F  F  F  F  F
F  F  F  F  F  F  F  F  F  F  F
F  F  F  F  F  F  F  F  F  F  F
F  F  H  F  F 